In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import random
import numpy as np
import copy
import matplotlib.pyplot as plt


import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [3]:
# Set device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [60]:
from torch.utils.cpp_extension import load
# Load and compile CUDA extension
maxplus_conv2d = load(
    name="maxplus_conv2d",
    sources=["conv2d_cuda.cu"],
    # extra_cuda_cflags=["-lineinfo"],  # Optional: Debugging info
    verbose=True
)

from torch.autograd import Function    
class MaxPlusConv2dFunction(Function):
    @staticmethod
    def forward(ctx, input, weight, bias, stride, padding):
        output, argmax_input_idx, argmax_weight_idx = maxplus_conv2d.maxplus_conv2d_forward(
            input, weight, bias, stride, padding
        )
        ctx.save_for_backward(argmax_input_idx, argmax_weight_idx)
        ctx.input_shape = input.shape
        ctx.weight_shape = weight.shape
        ctx.stride = stride
        ctx.padding = padding
        return output

    @staticmethod
    def backward(ctx, grad_output):
        argmax_input_idx, argmax_weight_idx = ctx.saved_tensors
        B, C_in, H_in, W_in = ctx.input_shape
        C_out, _, K, _ = ctx.weight_shape
        stride = ctx.stride
        padding = ctx.padding
        H_out = (H_in + 2 * padding - K) // stride + 1
        W_out = (W_in + 2 * padding - K) // stride + 1

        grad_input, grad_weight, grad_bias = maxplus_conv2d.maxplus_conv2d_backward(
            grad_output, argmax_input_idx, argmax_weight_idx,
            B, C_in, C_out, H_in, W_in, H_out, W_out, K
        )
        return grad_input, grad_weight, grad_bias, None, None

# Convenience wrapper
def maxplus_conv2d_wrapper(input, weight, bias=None, stride=1, padding=0):
    return MaxPlusConv2dFunction.apply(input, weight, bias, stride, padding)

class MorphConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=(3,3), stride=(1,1), padding=(1,1), bias=True, alpha=1.0):
        super(MorphConv2d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.bias = bias
        self.alpha = alpha

        self.weight = nn.Parameter(
            torch.normal(mean=0.0, std=alpha, size=(out_channels, in_channels, kernel_size[0], kernel_size[1]))
        )
        if bias:
            self.b = nn.Parameter(torch.normal(mean=0.0, std=1.0, size=(out_channels, )))
            self.b2 = nn.Parameter(torch.normal(mean=0.0, std=1.0, size=(out_channels, )))
        else:
            self.b = nn.Parameter(torch.zeros((out_channels, )) - 1e9)
            self.b2 = nn.Parameter(torch.zeros((out_channels, )) + 1e9)

    def forward(self, x):
        # Max morphological operation
        max_w = self.weight
        x_max = maxplus_conv2d_wrapper(x, max_w, self.b, self.stride[0], self.padding)

        # Min morphological operation
        min_w = self.weight
        x_min = -x
        x_min = maxplus_conv2d_wrapper(x_min, -min_w, -self.b2, self.stride[0], self.padding)
        x_min = -x_min

        # Aggregation of max and min operations
        x_out = (x_max + x_min)/2

        return x_out
    
class ConvLinAct(nn.Module):
    def __init__(self, channels, method="simple"):
        super(ConvLinAct, self).__init__()
        self.method = method

        if method == 'simple':
            self.a = nn.Parameter(torch.zeros(3, 3, channels))
            self.a.data[1,1,:] += 1
        else:
            tmp = nn.Conv2d(channels, channels, 3, bias = False).weight.data.detach()
            tmp = torch.transpose(tmp, 0, 2)
            tmp = torch.transpose(tmp, 1, 3)
            U, S, Vh = np.linalg.svd(tmp.cpu().numpy(), full_matrices=True)
            self.U = torch.tensor(U).to(device)
            self.Vh = torch.tensor(Vh).to(device)
            self.a = torch.zeros(3, 3, channels).to(device)
            self.a[:, :, :S.shape[2]] = torch.tensor(S)
            self.a = nn.Parameter(self.a)
    def forward(self, x):
        if self.method == 'simple':
            # x = x * self.a.unsqueeze(-1).unsqueeze(-1).unsqueeze(0).repeat(x.size(0), 1, x.size(2), x.size(3))
            tmp = torch.diag_embed(self.a)
            tmp = torch.transpose(tmp, 0, 2)
            tmp = torch.transpose(tmp, 1, 3)
            x = nn.functional.conv2d(x, tmp, padding=1)
        else:
            tmp = self.U @ (self.a.unsqueeze(-1) * self.Vh)
            tmp = torch.transpose(tmp, 0, 2)
            tmp = torch.transpose(tmp, 1, 3)
            x = nn.functional.conv2d(x, tmp, padding=1)
        return x

class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, kernel_size=1, stride=1, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNet20(nn.Module):
    def __init__(self, in_channels=3, num_blocks=[3, 3, 3], num_classes=10):
        super().__init__()
        self.in_planes = 16
        self.conv1 = nn.Conv2d(in_channels, 16, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        self.layer1 = self._make_layer(16, num_blocks[0])
        self.pool1 = nn.MaxPool2d(2, 2)

        self.layer2 = self._make_layer(32, num_blocks[1])
        self.pool2 = nn.MaxPool2d(2, 2)

        self.layer3 = self._make_layer(64, num_blocks[2])
        
        self.fc = nn.Linear(64, num_classes)

    def _make_layer(self, planes, num_blocks):
        layers = []
        for _ in range(num_blocks):
            layers.append(BasicBlock(self.in_planes, planes))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))

        out = self.layer1(out)
        out = self.pool1(out)

        out = self.layer2(out)
        out = self.pool2(out)

        out = self.layer3(out)
        
        out = F.avg_pool2d(out, out.shape[2]) 
        out = out.view(out.size(0), -1)
        return self.fc(out)  


class MPM_BasicBlock(nn.Module):
    def __init__(self, in_planes, planes):
        super().__init__()
        self.conv1 = MorphConv2d(in_planes, planes, kernel_size=(3,3), padding=1, bias=False)
        self.linact1 = ConvLinAct(planes, method="simple")
        # self.bn1 = nn.BatchNorm2d(planes)
        self.bn1 = nn.Identity()
        self.conv2 = MorphConv2d(planes, planes, kernel_size=(3,3), padding=1, bias=False)
        self.linact2 = ConvLinAct(planes, method="simple")
        # self.bn2 = nn.BatchNorm2d(planes)
        self.bn2 = nn.Identity()

        self.shortcut = nn.Sequential()
        if in_planes != planes:
            self.shortcut = MorphConv2d(in_planes, planes, kernel_size=(1,1), padding=0, bias=False)

    def forward(self, x):
        out = self.linact1(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.linact2(out)
        return out

class MPM_ResNet20(nn.Module):
    def __init__(self, in_channels=3, num_blocks=[3,3,3], num_classes=10):
        super().__init__()
        self.in_planes = 16

        self.conv1 = MorphConv2d(in_channels, 16, kernel_size=(3, 3), padding=1, bias=False)
        self.linact1 = ConvLinAct(16, method = "simple")
        # self.bn1 = nn.BatchNorm2d(16)
        self.bn1 = nn.Identity()

        self.layer1 = self._make_layer(16, num_blocks[0])
        self.pool1 = nn.MaxPool2d(2, 2)

        self.layer2 = self._make_layer(32, num_blocks[1])
        self.pool2 = nn.MaxPool2d(2, 2)

        self.layer3 = self._make_layer(64, num_blocks[2])
        
        self.fc = nn.Linear(64, num_classes)

    def _make_layer(self, planes, num_blocks):
        layers = []
        for _ in range(num_blocks):
            layers.append(MPM_BasicBlock(self.in_planes, planes))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.linact1(self.bn1(self.conv1(x)))

        out = self.layer1(out)
        out = self.pool1(out)

        out = self.layer2(out)
        out = self.pool2(out)

        out = self.layer3(out)
        
        out = F.avg_pool2d(out, out.shape[2]) 
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out  

Using /home/anonymous/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module maxplus_conv2d, skipping build step...
Loading extension module maxplus_conv2d...


In [5]:
def train(model, criterion, optimizer, train_loader, val_loader, num_epochs=50, return_list=False):
    # Training and validation loop
    best_val_accuracy = 0.0
    best_model = None

    train_list = []
    val_list = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Validation phase
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total
        train_list.append(train_accuracy)
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = 100 * correct / total
        val_list.append(val_accuracy)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Train Accuracy: {train_accuracy:.2f}%, Validation Accuracy: {val_accuracy:.2f}%")

        # Save best model based on validation accuracy
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model = copy.deepcopy(model)

    if return_list:
        return best_model, train_list, val_list
    else:
        return best_model

def test(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy on the test set: {accuracy:.2f}%')

In [6]:
# Load and preprocess MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))])

full_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

# Split train dataset into training and validation sets
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

Files already downloaded and verified
Files already downloaded and verified


In [ ]:
# Initialize model, loss function, and optimizer
model = ResNet20(in_channels=3).to(device)

total_params = 0
for param in model.parameters():
    total_params += param.numel()
print(f"Total number of parameters: {total_params}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 272282
Epoch [1/50], Loss: 1.0102, Train Accuracy: 55.81%, Validation Accuracy: 55.42%
Epoch [2/50], Loss: 0.9572, Train Accuracy: 67.36%, Validation Accuracy: 65.30%
Epoch [3/50], Loss: 0.9953, Train Accuracy: 74.09%, Validation Accuracy: 70.80%
Epoch [4/50], Loss: 0.7005, Train Accuracy: 80.22%, Validation Accuracy: 76.01%
Epoch [5/50], Loss: 0.4663, Train Accuracy: 79.75%, Validation Accuracy: 74.74%
Epoch [6/50], Loss: 0.5554, Train Accuracy: 85.45%, Validation Accuracy: 78.75%
Epoch [7/50], Loss: 0.5123, Train Accuracy: 86.33%, Validation Accuracy: 78.76%
Epoch [8/50], Loss: 0.3781, Train Accuracy: 88.92%, Validation Accuracy: 80.16%
Epoch [9/50], Loss: 0.2935, Train Accuracy: 89.00%, Validation Accuracy: 79.26%
Epoch [10/50], Loss: 0.3729, Train Accuracy: 91.68%, Validation Accuracy: 81.00%
Epoch [11/50], Loss: 0.2389, Train Accuracy: 88.99%, Validation Accuracy: 77.99%
Epoch [12/50], Loss: 0.1664, Train Accuracy: 91.10%, Validation Accuracy: 80.27%
Ep

In [30]:
test(model, test_loader)

Accuracy on the test set: 82.04%


In [ ]:
# Initialize model, loss function, and optimizer
model = MPM_ResNet20(in_channels=3).to(device)

total_params = 0
for param in model.parameters():
    total_params += param.numel()
print(f"Total number of parameters: {total_params}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader, num_epochs=100)

Total number of parameters: 278666
Epoch [1/100], Loss: 2.1081, Train Accuracy: 21.50%, Validation Accuracy: 20.79%
Epoch [2/100], Loss: 2.1236, Train Accuracy: 26.98%, Validation Accuracy: 26.17%
Epoch [3/100], Loss: 1.9627, Train Accuracy: 30.25%, Validation Accuracy: 29.61%
Epoch [4/100], Loss: 1.9266, Train Accuracy: 31.36%, Validation Accuracy: 30.69%
Epoch [5/100], Loss: 1.7275, Train Accuracy: 34.14%, Validation Accuracy: 33.80%
Epoch [6/100], Loss: 1.8368, Train Accuracy: 35.90%, Validation Accuracy: 35.82%
Epoch [7/100], Loss: 1.6267, Train Accuracy: 39.25%, Validation Accuracy: 38.33%
Epoch [8/100], Loss: 1.6716, Train Accuracy: 42.45%, Validation Accuracy: 41.87%
Epoch [9/100], Loss: 1.7057, Train Accuracy: 44.05%, Validation Accuracy: 43.25%
Epoch [10/100], Loss: 1.5535, Train Accuracy: 47.82%, Validation Accuracy: 47.26%
Epoch [11/100], Loss: 1.4346, Train Accuracy: 49.83%, Validation Accuracy: 49.14%
Epoch [12/100], Loss: 1.2754, Train Accuracy: 50.69%, Validation Accurac

In [57]:
test(model, test_loader)

Accuracy on the test set: 61.96%


In [58]:
# Load and preprocess MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

full_train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.FashionMNIST(root='./data', train=False, transform=transform, download=True)

# Split train dataset into training and validation sets
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [61]:
# Initialize model, loss function, and optimizer
model = ResNet20(in_channels=1).to(device)

total_params = 0
for param in model.parameters():
    total_params += param.numel()
print(f"Total number of parameters: {total_params}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 271994
Epoch [1/50], Loss: 0.1700, Train Accuracy: 86.65%, Validation Accuracy: 85.66%
Epoch [2/50], Loss: 0.1546, Train Accuracy: 89.99%, Validation Accuracy: 88.57%
Epoch [3/50], Loss: 0.2530, Train Accuracy: 92.13%, Validation Accuracy: 90.88%
Epoch [4/50], Loss: 0.2360, Train Accuracy: 93.72%, Validation Accuracy: 92.36%
Epoch [5/50], Loss: 0.0641, Train Accuracy: 94.12%, Validation Accuracy: 92.22%
Epoch [6/50], Loss: 0.0945, Train Accuracy: 93.78%, Validation Accuracy: 91.62%
Epoch [7/50], Loss: 0.2016, Train Accuracy: 93.12%, Validation Accuracy: 90.78%
Epoch [8/50], Loss: 0.1307, Train Accuracy: 94.51%, Validation Accuracy: 91.54%
Epoch [9/50], Loss: 0.0644, Train Accuracy: 96.16%, Validation Accuracy: 92.65%
Epoch [10/50], Loss: 0.0528, Train Accuracy: 96.65%, Validation Accuracy: 92.83%
Epoch [11/50], Loss: 0.2462, Train Accuracy: 96.94%, Validation Accuracy: 92.68%
Epoch [12/50], Loss: 0.1468, Train Accuracy: 96.40%, Validation Accuracy: 91.80%
Ep

In [62]:
test(model, test_loader)

Accuracy on the test set: 92.43%


In [63]:
# Initialize model, loss function, and optimizer
model = MPM_ResNet20(in_channels=1).to(device)

total_params = 0
for param in model.parameters():
    total_params += param.numel()
print(f"Total number of parameters: {total_params}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 278378
Epoch [1/50], Loss: 1.2655, Train Accuracy: 62.71%, Validation Accuracy: 62.23%
Epoch [2/50], Loss: 0.8069, Train Accuracy: 70.90%, Validation Accuracy: 70.59%
Epoch [3/50], Loss: 0.6713, Train Accuracy: 77.70%, Validation Accuracy: 76.98%
Epoch [4/50], Loss: 0.6765, Train Accuracy: 79.35%, Validation Accuracy: 78.47%
Epoch [5/50], Loss: 0.4721, Train Accuracy: 80.85%, Validation Accuracy: 79.95%
Epoch [6/50], Loss: 0.2905, Train Accuracy: 83.84%, Validation Accuracy: 82.94%
Epoch [7/50], Loss: 0.3953, Train Accuracy: 84.24%, Validation Accuracy: 83.72%
Epoch [8/50], Loss: 0.6704, Train Accuracy: 86.06%, Validation Accuracy: 85.28%
Epoch [9/50], Loss: 0.3659, Train Accuracy: 83.51%, Validation Accuracy: 82.76%
Epoch [10/50], Loss: 0.3269, Train Accuracy: 87.20%, Validation Accuracy: 86.13%
Epoch [11/50], Loss: 0.5544, Train Accuracy: 87.41%, Validation Accuracy: 86.37%
Epoch [12/50], Loss: 0.3247, Train Accuracy: 88.33%, Validation Accuracy: 87.22%
Ep

In [64]:
test(model, test_loader)

Accuracy on the test set: 89.49%
